In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

A_DIR = Path("") # load training set A
B_DIR = Path("") # load training set B

ARCHIVE_DIR = A_DIR.parents[1]   # .../archive
assert ARCHIVE_DIR == B_DIR.parents[1], f"A/B not under same 'archive' dir:\n{ARCHIVE_DIR}\n{B_DIR.parents[1]}"

######### FIND .psv FILES #########
filesA = sorted(A_DIR.rglob("p*.psv"))
filesB = sorted(B_DIR.rglob("p*.psv"))
print(f"Found {len(filesA)} files in A, {len(filesB)} files in B")
if not filesA and not filesB:
    raise FileNotFoundError("No .psv files found. Double-check A_DIR/B_DIR paths.")

###########  HELPER: READ ONE FILE, ADD PatientID(A/B), Hour(0..n-1), KEEP ORIGINAL COL ORDER #######
def read_one(fpath: Path, hospital_tag: str) -> pd.DataFrame:
    g = pd.read_csv(fpath, sep="|", low_memory=False)
    original_cols = list(g.columns)  # preserve raw column order

    # Sort time
    time_col = "ICULOS" if "ICULOS" in g.columns else ("Hour" if "Hour" in g.columns else None)
    if time_col is not None:
        g = g.sort_values(time_col).reset_index(drop=True)

    # If raw files already had a 'Hour' column, preserve it as 'Hour_orig'
    if "Hour" in g.columns:
        g = g.rename(columns={"Hour": "Hour_orig"})
        original_cols = ["Hour_orig" if c == "Hour" else c for c in original_cols]

    # New per-patient Hour: 0..n-1 (resets for each patient)
    g["Hour"] = range(len(g))

    # Prefixed PatientID from filename, e.g., 'p000664' -> 'A000664' or 'B000664'
    stem = fpath.stem
    digits = "".join(ch for ch in stem if ch.isdigit())
    g["PatientID"] = f"{hospital_tag}{digits}"

    # Hospital tag
    g["Hospital"] = hospital_tag

    # Reorder columns: PatientID, Hour, then original columns (preserved order), then Hospital
    desired_cols = ["PatientID", "Hour"] + original_cols + ["Hospital"]
    seen = set()
    ordered_cols = [c for c in desired_cols if (c not in seen and not seen.add(c)) and c in g.columns]

    return g[ordered_cols]

############ CONCATENATE ALL PATIENTS ###########
dfs = [read_one(f, "A") for f in filesA] + [read_one(f, "B") for f in filesB]
df = pd.concat(dfs, ignore_index=True)

# Final sort (by PatientID then Hour)
df = df.sort_values(["PatientID", "Hour"]).reset_index(drop=True)

# Quick sanity prints
print("Rows, Cols:", df.shape)
print("First 10 columns:", df.columns[:10].tolist())
print("Unique patients (total):", df["PatientID"].nunique())
if "Hospital" in df.columns:
    print(df.groupby("Hospital")["PatientID"].nunique().rename("patients_per_site"))

############# SAVE TO archive/processed_data/ ###########
out_dir = ARCHIVE_DIR / "processed_data"
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = out_dir / "all_patients.parquet"
csv_path     = out_dir / "all_patients.csv"

# Parquet (fast, typed) — requires pyarrow or fastparquet
df.to_parquet(parquet_path, index=False)
print("Saved Parquet →", parquet_path)

# CSV (human-readable; larger)
df.to_csv(csv_path, index=False)
print("Saved CSV →", csv_path)

# (Optional) compressed CSV
gz_path = csv_path.with_suffix(".csv.gz")
df.to_csv(gz_path, index=False, compression="gzip")
print("Saved compressed CSV →", gz_path)


In [ ]:
ARCHIVE_DIR = Path(".../processed_data")
SRC = ARCHIVE_DIR / "all_patients.csv"
DST = ARCHIVE_DIR / "all_patients2.csv"

df = pd.read_csv(SRC, low_memory=False)
for c in ["PatientID", "Hour"]:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# columns to process
vitals_like = [c for c in ["Temp", "O2Sat", "HR"] if c in df.columns]
labs_like = [c for c in ["Resp", "pH", "PaCo2", "SaO2", "AST", "BUN",
    "Calcium", "Chloride", "Creatinine", "Glucose",
    "Magnesium", "Phosphate", "Potassium",
    "Hct", "Hgb", "WBC", "Platelets"] if c in df.columns]

cols_to_process = vitals_like + labs_like
if not cols_to_process:
    raise ValueError("None of the expected columns to process were found.")

# keep raw copies only for the columns we modify
for c in cols_to_process:
    raw_col = f"{c}_raw"
    if raw_col not in df.columns:
        df[raw_col] = df[c]

# stable row id to restore original order
df["__orig_idx__"] = np.arange(len(df), dtype=np.int64)

def correct_ast_values(s: pd.Series) -> pd.Series:
    """
    If AST has a 4-digit value (1000..9999), divide by 100.
    Leaves other values unchanged. Works regardless of hour.
    """
    s_num = pd.to_numeric(s, errors="coerce")
    mask = (s_num >= 1000) & (s_num <= 9999)
    s_num.loc[mask] = s_num.loc[mask] / 100.0
    # preserve original dtype-ish (float is fine for labs)
    return s_num

def fill_series_start_at1(s: pd.Series, hours: pd.Series) -> pd.Series:
    """
    Piecewise-linear fill used for Temp/O2Sat/HR and labs:
      - linear interpolation ONLY between two known points (internal gaps)
      - then bfill/ffill for leading/trailing gaps
    Critically: preserves Hour==0 EXACTLY as originally loaded.
    """
    s_num = pd.to_numeric(s, errors="coerce")

    # fill globally so hour 0 can serve as a boundary, then restore hour 0
    filled = s_num.interpolate(method="linear", limit_area="inside")
    filled = filled.bfill().ffill()

    # restore hour 0 to its original value
    h = pd.to_numeric(hours, errors="coerce")
    h0_mask = (h == 0)
    filled[h0_mask] = s_num[h0_mask]

    return filled

def process_one_patient(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("Hour").copy()
    hrs = g["Hour"]

    # Apply AST value correction BEFORE any filling
    if "AST" in g.columns:
        g["AST"] = correct_ast_values(g["AST"])

    # Apply the common fill rule to all requested columns
    for c in cols_to_process:
        g[c] = fill_series_start_at1(g[c], hrs)

    return g

# explicit loop over groups (keeps PatientID; avoids groupby.apply warnings)
pieces = []
for pid, g in df.groupby("PatientID", sort=False):
    pieces.append(process_one_patient(g))

df_out = pd.concat(pieces, ignore_index=False)

# restore original row order exactly as in source file
df_out = df_out.sort_values("__orig_idx__").drop(columns="__orig_idx__")

# save
df_out.to_csv(DST, index=False)
print(f"Saved → {DST}")